# Homework 6.1 - Step 10

## LLM Critic for the Attention Plots

This notebook does **not** write the final human critique.

Its purpose is only to:

1. collect the fixed numerical evidence;
2. identify the saved attention plots;
3. construct the question for the LLM critic;
4. provide a place to paste the LLM response;
5. leave the student's assessment section blank.

The numerical experiment should already be complete before this notebook is used.

In [1]:
from pathlib import Path
import json

import pandas as pd

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "codes" else cwd

DATA_DIR = PROJECT_ROOT / "data" / "hw6_1"
RESULT_DIR = PROJECT_ROOT / "result images" / "hw6_1"

required = {
    "markov_test": (
        DATA_DIR
        / "task3_markov_test_results.csv"
    ),
    "attention_validation": (
        DATA_DIR
        / "task4_5_9_attention_validation_results.csv"
    ),
    "attention_test": (
        DATA_DIR
        / "task4_5_9_attention_test_results.csv"
    ),
    "attention_selection": (
        DATA_DIR
        / "task4_5_9_selected_attention_model.json"
    ),
    "comparison": (
        DATA_DIR
        / "task5_attention_vs_markov_common_support.csv"
    ),
    "attention_common_support": (
        DATA_DIR
        / "task9_attention_common_support_results.csv"
    ),
    "switching_postprefix": (
        DATA_DIR
        / "task7_lobe_switching_postprefix_summary.csv"
    ),
    "attention_lag_summary": (
        DATA_DIR
        / "task6_attention_lag_summary.csv"
    ),
    "support_comparison": (
        DATA_DIR
        / "task7_support_comparison.csv"
    ),
    "switching_summary": (
        DATA_DIR
        / "task7_lobe_switching_summary.csv"
    ),
}

for name, path in required.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {name}: {path}\n"
            "Run Steps 1-9 before Step 10."
        )

markov_test = pd.read_csv(
    required["markov_test"]
)

attention_validation = pd.read_csv(
    required[
        "attention_validation"
    ]
)

attention_test = pd.read_csv(
    required["attention_test"]
)

attention_selection = json.loads(
    required[
        "attention_selection"
    ].read_text(
        encoding="utf-8"
    )
)

comparison = pd.read_csv(
    required["comparison"]
)

attention_common_support = pd.read_csv(
    required["attention_common_support"]
)

switching_postprefix_text = required["switching_postprefix"].read_text(encoding="utf-8")

attention_lag_summary = pd.read_csv(
    required[
        "attention_lag_summary"
    ]
)

support_comparison = pd.read_csv(
    required[
        "support_comparison"
    ]
)

switching_summary_text = (
    required[
        "switching_summary"
    ].read_text(
        encoding="utf-8"
    )
)

print("All fixed evidence loaded.")

All fixed evidence loaded.


## Attention plots to show the critic

Show the LLM the following saved figures together with the prompt:

- `step6_attention_matrix_near_lobe_switch.png`
- `step6_attention_profile_near_lobe_switch.png`
- `step6_attention_matrix_within_one_lobe.png`
- `step6_attention_profile_within_one_lobe.png`
- `step7_true_markov_attention_xz.png`

The plots should be interpreted together with the numerical evidence rather than by visual appearance alone.

In [2]:
summary_parts = [
    "SYSTEM:",
    (
        "Classical Lorenz system with "
        "sigma=10, rho=28, beta=8/3."
    ),
    (
        "The system is deterministic, "
        "dissipative, and chaotic."
    ),
    "",
    "MARKOV TEST RESULTS:",
    markov_test.to_string(index=False),
    "",
    "ATTENTION VALIDATION RESULTS:",
    attention_validation.to_string(index=False),
    "",
    "ATTENTION TEST RESULTS:",
    attention_test.to_string(index=False),
    "",
    "VALIDATION-SELECTED ATTENTION MODEL:",
    json.dumps(
        attention_selection,
        indent=2,
    ),
    "",
    "COMMON-SUPPORT ATTENTION RESULTS:",
    attention_common_support.to_string(index=False),
    "",
    "SELECTED MODEL COMPARISON ON COMMON TARGET POSITIONS:",
    comparison.to_string(index=False),
    "",
    "ATTENTION-LAG SUMMARY:",
    attention_lag_summary.to_string(index=False),
    "",
    "GENERATED SUPPORT COMPARISON:",
    support_comparison.to_string(index=False),
    "",
    "LOBE-SWITCHING SUMMARY (COMMON 32-TOKEN PREFIX):",
    switching_summary_text,
    "",
    "LOBE-SWITCHING SUMMARY AFTER REMOVING THE TRUE PREFIX:",
    switching_postprefix_text,
]

fixed_evidence = "\n".join(
    summary_parts
)

evidence_path = (
    DATA_DIR
    / "task10_fixed_evidence_summary.txt"
)

evidence_path.write_text(
    fixed_evidence,
    encoding="utf-8",
)

print(fixed_evidence)
print()
print("Saved:", evidence_path.resolve())

SYSTEM:
Classical Lorenz system with sigma=10, rho=28, beta=8/3.
The system is deterministic, dissipative, and chaotic.

MARKOV TEST RESULTS:
 context_length_m  evaluated_positions  accuracy  exact_history_coverage  unseen_exact_history_fraction  selected_by_validation
                1                12500  0.615440                0.999840                       0.000160                   False
                2                12490  0.698479                0.997998                       0.002002                   False
                3                12480  0.758173                0.994712                       0.005288                    True

ATTENTION VALIDATION RESULTS:
 context_length  trainable_parameters  best_epoch  validation_nll  validation_accuracy  training_seconds                                          checkpoint
              4                186576          10        0.598779             0.768885         13.685280 data\hw6_1\attention_models\attention_context_04.pt
 

# Question for the LLM Critic

Use the question below after showing the LLM the selected attention plots and the fixed numerical evidence.

The critic is asked to distinguish genuine dynamical evidence from attractive-looking attention patterns.

In [3]:
critic_question = "\n".join([
    "You are the critic for a Math 638 experiment on causal attention for tokenized Lorenz trajectories.",
    "",
    "The numerical experiment is already complete. Do not redesign the experiment and do not invent new measurements.",
    "",
    "You are given:",
    "- the Markov baseline results;",
    "- attention-model validation and test results;",
    "- the selected attention context length;",
    "- attention heatmaps and final-query attention profiles for a within-lobe example and a near-lobe-switch example;",
    "- quantitative expected-attention-lag summaries;",
    "- coarse attractor-support statistics for true, Markov-generated, and attention-generated token trajectories;",
    "- lobe-switching statistics for those trajectories.",
    "",
    "Answer the following questions.",
    "",
    "1. What do the attention plots appear to focus on?",
    "2. Does the model mostly attend to recent tokens, or does it place substantial weight on earlier parts of the context?",
    "3. Is there evidence that the attention pattern changes near a Lorenz lobe switch compared with ordinary within-lobe motion?",
    "4. Do the numerical attention-lag summaries support the visual impression from the plots?",
    "5. Does the attention model improve next-token prediction over the validation-selected Markov / m-gram baseline?",
    "6. Does longer attention context improve prediction throughout the tested range, or is there a point of diminishing or negative returns?",
    "7. Do the attention weights provide evidence of genuine dynamical structure, or could they simply reflect token-frequency and short-term predictive correlations?",
    "8. Does the attention-generated model reproduce coarse Lorenz behavior better than the Markov generator according to the reported support and lobe-switching diagnostics?",
    "9. Which conclusions are strongly supported by the experiment?",
    "10. Which conclusions would exceed the evidence?",
    "",
    "Important constraints:",
    "- The selected system is the dissipative Lorenz system, not the Hamiltonian Kirchhoff system.",
    "- Do not invent conserved Hamiltonian energy, momentum, or Casimir quantities.",
    "- Do not treat a visually attractive attention heatmap as proof that attention has discovered a physical law.",
    "- Do not treat next-token accuracy as proof that the model has recovered the continuous Lorenz vector field.",
    "- Distinguish measured evidence from interpretation and speculation.",
    "",
    "Return:",
    "A. Evidence-supported interpretation.",
    "B. Claims that would exceed the evidence.",
])

prompt = (
    critic_question
    + "\n\nFIXED NUMERICAL EVIDENCE:\n\n"
    + fixed_evidence
)

prompt_path = (
    DATA_DIR
    / "task10_llm_critic_prompt.txt"
)

prompt_path.write_text(
    prompt,
    encoding="utf-8",
)

print(prompt)
print()
print("Saved:", prompt_path.resolve())

You are the critic for a Math 638 experiment on causal attention for tokenized Lorenz trajectories.

The numerical experiment is already complete. Do not redesign the experiment and do not invent new measurements.

You are given:
- the Markov baseline results;
- attention-model validation and test results;
- the selected attention context length;
- attention heatmaps and final-query attention profiles for a within-lobe example and a near-lobe-switch example;
- quantitative expected-attention-lag summaries;
- coarse attractor-support statistics for true, Markov-generated, and attention-generated token trajectories;
- lobe-switching statistics for those trajectories.

Answer the following questions.

1. What do the attention plots appear to focus on?
2. Does the model mostly attend to recent tokens, or does it place substantial weight on earlier parts of the context?
3. Is there evidence that the attention pattern changes near a Lorenz lobe switch compared with ordinary within-lobe motio

## LLM Response

Paste the LLM response below **exactly as received**.

Do not rewrite the response before you assess it.

In [4]:
LLM_RESPONSE = r"""A. Evidence-supported interpretation

1. **What the attention plots focus on.** The final-query attention is sparse and multimodal rather than concentrated only on the immediately previous token. In the selected near-switch example, the largest weights occur at lags 27 (about 0.213), 15 (about 0.181), and 26 (about 0.149). In the selected within-lobe example, the strongest weights occur at lags 8 (about 0.289), 6 (about 0.254), and 7 (about 0.116), with a secondary weight near lag 25 (about 0.091). Thus the model uses nonlocal positions within the 32-token context.

2. **Recent versus earlier context.** The model does not attend only to the recent past. Across the test set, the mean expected attention lag is about 13.25 steps for non-switch targets and 16.26 steps for lobe-switch targets. With dt=0.02, these correspond to about 0.265 and 0.325 time units. This is evidence that appreciable attention mass is often placed well back in the available context.

3. **Change near lobe switches.** There is a reproducible association between switch targets and longer attention lag. The average lag increases by about 3.01 token steps near switches, and the switch-target mean exceeds the non-switch mean in each of the 10 test books. The two plotted examples are consistent with that aggregate result. This supports the statement that the learned attention pattern changes near lobe transitions, but it does not identify a physical mechanism by itself.

4. **Do the quantitative summaries support the plots?** Yes. The plots suggest different temporal emphasis near a switch, and the all-target lag summary supports that observation. However, attention weights are not causal explanations; token identity, position embeddings, recurrent geometric patterns in the attractor, and training correlations can all influence the weights.

5. **Attention versus the Markov baseline.** On the same held-out target positions (all targets at position 32 or later), the validation-selected m=3 Markov model has test accuracy 0.758737, while the selected L=32 attention model has test accuracy 0.839212. The improvement is about 0.0805, or 8.05 percentage points. Therefore the fitted attention predictor is clearly better for one-step token prediction in this experiment. This comparison does not isolate 'attention alone,' because the attention model also has a longer context and many trainable parameters.

6. **Effect of longer attention context.** Longer context helps strongly from L=4 to L=8 and still helps from L=8 to L=16. On common validation targets, accuracy changes from 0.768909 to 0.827564 to 0.836916 to 0.838228 for L=4,8,16,32, while NLL changes from 0.598656 to 0.460849 to 0.425505 to 0.424146. Thus there are clear diminishing returns by L=16 to 32. No negative return is observed within the tested range, but contexts longer than 32 were not tested. Also, the selected L=32 checkpoint occurs at the maximum training epoch, so the experiment does not establish that optimization behavior for still longer contexts would remain easy.

7. **Do attention weights prove genuine dynamics?** No. They provide evidence of a learned temporal association related to the tokenized dynamics, especially because switch targets have systematically longer expected lag. But the experiment cannot distinguish a physically meaningful dynamical feature from correlations caused by token frequency, discretized geometry, or repeated local trajectory motifs. Attention weights should therefore be treated as a diagnostic of model dependence, not as a discovered physical law.

8. **Long-run generation.** The attention generator is not better than the Markov generator on the reported coarse Lorenz diagnostics. All three datasets have roughly 95% of box-center states within the validation-derived support threshold: true test 0.9511, Markov 0.9520, and attention 0.9479. That support test is weak and does not measure the invariant distribution. More importantly, true test trajectories switch at about 10.96 times per 1000 steps, the corrected Markov generator at about 10.76, and the attention generator at about 25.22. Mean lobe residence time is about 1.78 for truth, 1.79 for Markov, and only 0.85 for attention. Removing the common 32-token true prefix leaves the same conclusion. Thus the attention model has better one-step accuracy but substantially worse long-run switching statistics.

9. **Strongly supported conclusions.** (i) causal attention improves held-out one-step token prediction for this Lorenz dataset and discretization; (ii) useful gains from context length diminish after about L=16 in the tested range; (iii) the model frequently assigns weight to nonrecent positions; (iv) lobe-switch targets are associated with longer expected attention lag; and (v) superior one-step accuracy does not imply superior autonomous long-run dynamics.

10. **Claims that would exceed the evidence.** The experiment does not show that the model recovered the Lorenz differential equations, identified the continuous vector field, discovered a conserved quantity, learned a causal physical mechanism, or reproduced the true Lorenz invariant measure or Lyapunov behavior. It also does not establish that attention would outperform the baseline for other dynamical systems, other discretizations, or other training-set sizes, because those factors were not varied.

B. Claims that would exceed the evidence

The strongest overclaim would be to interpret visually attractive attention patterns as proof that the network discovered Lorenz physics. The experiment instead supports a narrower statement: attention learns useful nonlocal predictive correlations in the tokenized trajectories, and those correlations change systematically near lobe switches. The long-run generation failure shows why prediction accuracy and physical fidelity must be evaluated separately."""

response_path = (
    DATA_DIR
    / "task10_llm_critic_response.txt"
)

response_path.write_text(
    LLM_RESPONSE,
    encoding="utf-8",
)

print(LLM_RESPONSE)
print()
print("Saved:", response_path.resolve())

A. Evidence-supported interpretation

1. **What the attention plots focus on.** The final-query attention is sparse and multimodal rather than concentrated only on the immediately previous token. In the selected near-switch example, the largest weights occur at lags 27 (about 0.213), 15 (about 0.181), and 26 (about 0.149). In the selected within-lobe example, the strongest weights occur at lags 8 (about 0.289), 6 (about 0.254), and 7 (about 0.116), with a secondary weight near lag 25 (about 0.091). Thus the model uses nonlocal positions within the 32-token context.

2. **Recent versus earlier context.** The model does not attend only to the recent past. Across the test set, the mean expected attention lag is about 13.25 steps for non-switch targets and 16.26 steps for lobe-switch targets. With dt=0.02, these correspond to about 0.265 and 0.325 time units. This is evidence that appreciable attention mass is often placed well back in the available context.

3. **Change near lobe switches

# My Assessment of the LLM Response

The LLM response is mostly careful and is consistent with the fixed numerical evidence. It correctly avoids treating the Lorenz system as Hamiltonian, does not invent energy or Casimir invariants, and distinguishes one-step prediction from autonomous long-run generation. Its most useful point is the contrast between the attention model's higher next-token accuracy and its much worse lobe-switching statistics. That contrast directly shows why a sequence model can be locally predictive without reproducing the long-run dynamics.

I agree with the LLM that the attention pattern is not purely local. The selected profiles place substantial mass at nonrecent lags, and the all-target statistic is stronger evidence than either individual heatmap: the mean expected lag is about 13.25 for ordinary targets and 16.26 for switch targets. The difference is also positive in every held-out test book. This supports an association between lobe switching and longer attention reach.

However, I would weaken any wording that calls this a discovered dynamical mechanism. Attention weights are model-internal dependence weights, not causal attribution. The model may be exploiting repeated token patterns, the geometry created by the box discretization, position embeddings, or token-frequency correlations. The two hand-selected plots are illustrative rather than independent evidence; the aggregate lag calculation is the appropriate basis for the claim.

I would also qualify the statement that 'attention improves over Markov.' On common target positions the numerical result is clear: 83.921% versus 75.874% test accuracy. But this comparison does not isolate the attention operation alone. The selected attention model has a 32-token context and roughly 188k trainable parameters, while the baseline is an m=3 empirical model. Therefore the supported conclusion is that **this attention-based predictor configuration** performs better, not that attention is solely responsible for the entire gain.

The attractor-support diagnostic also needs cautious interpretation. Roughly 95% of true, Markov-generated, and attention-generated box centers lie within the validation-derived threshold, so the attention generator does not obviously leave the coarse training support. But this is a weak geometric check and cannot establish the correct invariant measure or chaotic dynamics. The lobe-switching statistic is much more discriminating here: the attention generator switches about 25.22 times per 1000 steps versus 10.96 for truth, whereas the corrected Markov generator is about 10.76.

Finally, the context-length experiment supports diminishing returns, not an unlimited benefit from longer memory. The gain from L=16 to L=32 is very small compared with the gain from L=4 to L=8. Because only one Lorenz discretization and one training-set size were tested, I would not generalize the conclusion to other systems, box resolutions, or data regimes.

**Final critique:** the LLM interpretation is useful when it stays close to the measured accuracy, lag, support, and switching statistics. The claims become too strong if the attention heatmaps are treated as physical explanations. The safest conclusion is that attention learns nonlocal predictive correlations that are measurably different near lobe switches, improves one-step prediction in this experiment, but does not by itself recover the Lorenz dynamics and actually performs worse than the Markov generator on long-run lobe-switching behavior.

## Step 10 outputs

- `task10_fixed_evidence_summary.txt`
- `task10_llm_critic_prompt.txt`
- `task10_llm_critic_response.txt`

The LLM response is recorded verbatim in the response file above, followed by a separate human critique in this notebook.